# Urgent rescoring: prediction23 holdout (2026-07-22)

Rescores IBS, the CIF-based readmission IBS and its competing-risk skill, the
corrected 3-arm readmission comparison, and threshold-based Sens/Spec/PPV/NPV
at fixed horizons -- all on the held-out 20% test set from the already-saved
predictions in `pred23_holdout_validation_2026_07_19.rds`, using the
corrected, order-preserving IPCW censoring lookup. No refit, no
re-imputation, no re-split, no other metrics (C-index, calibration, DCA,
NRI/IDI) recomputed.

Run once in a clean R session, top to bottom. Expect roughly 5 minutes end to
end. Every point IBS value was verified against a documented oracle (abs
diff < 1e-10) before this notebook was delivered; see the closing cell.

In [ ]:
# prediction23 holdout: urgent IBS rescoring (2026-07-22)
# Rescores IBS from stored held-out predictions with the corrected IPCW
# engine. No refit, no re-imputation, no re-split. Old values are read from
# the SAME loaded bundle (its own $summary, and any prior ibs_bp1/bp2/CIF
# objects) so before/after uses the exact same models/horizons -- no extra
# files are opened for the comparison.

suppressPackageStartupMessages(library(survival))

project_root <- "g:/My Drive/Alvacast/SISTRAT 2023"

bundle_file <- file.path(project_root, "data/20241015_out/pred23_holdout_validation_2026_07_19.rds")
out_fix <- file.path(project_root, "cons/_out/pred23_ibs_corrected_2026_07_22")
dir.create(out_fix, recursive = TRUE, showWarnings = FALSE)

cat("Loading bundle with load() (it is an .RData image, not a plain RDS object)...\n")
t0 <- Sys.time()
saved <- new.env(parent = globalenv())
load(bundle_file, envir = saved)
cat(sprintf("Loaded %d objects in %.1f min.\n", length(ls(saved)), as.numeric(difftime(Sys.time(), t0, units = "mins"))))

required_objects <- c("results_boot_val_bp1", "results_boot_val_bp2", "results_boot_val_bp1_cif", "results_boot_val_bp2_cif")
stopifnot(all(vapply(required_objects, exists, logical(1), envir = saved, inherits = FALSE)))

# ---- OLD (pre-fix) values, read directly from the loaded bundle: no extra load ----
old_from_summary <- function(obj, risk, horizon) {
  s <- obj$summary
  v <- s$mean[s$Metric == "IBS" & s$Risk == risk & s$Time == as.character(horizon)]
  if (length(v) == 0) NA_real_ else v[[1]]
}
HORIZONS <- c(6, 12, 36, 60)
# NOTE: risk is stored lowercase here ("readmission"/"death") to match the
# corrected engine's own $risk column (ibs_window_bootstrap_core()'s output),
# even though the bundle's own $summary$Risk uses Title Case ("Readmission").
old_dualscore <- rbind(
  data.frame(model = "readmission_shared", risk = "readmission", horizon = HORIZONS,
             ibs_old = vapply(HORIZONS, function(h) old_from_summary(saved$results_boot_val_bp1, "Readmission", h), numeric(1))),
  data.frame(model = "death_full_ph", risk = "death", horizon = HORIZONS,
             ibs_old = vapply(HORIZONS, function(h) old_from_summary(saved$results_boot_val_bp1, "Death", h), numeric(1))),
  data.frame(model = "death_shap", risk = "death", horizon = HORIZONS,
             ibs_old = vapply(HORIZONS, function(h) old_from_summary(saved$results_boot_val_bp2, "Death", h), numeric(1)))
)

# Bonus: if the bundle already has prior ibs_bp1/ibs_bp2/CIF bootstrap objects
# (same B/seed/methodology, just the old buggy engine), extract their point
# estimates too, for a same-methodology comparison. Defensive: skip silently
# if the structure does not match (does not block the main rescoring).
old_boot_point <- function(x, risk_val, horizon_val) {
  tryCatch({
    v <- x$point[x$risk == risk_val & x$horizon == horizon_val]
    if (length(v) == 0) NA_real_ else v[[1]]
  }, error = function(e) NA_real_)
}
have_old_boot <- exists("ibs_bp1", envir = saved, inherits = FALSE) && exists("ibs_bp2", envir = saved, inherits = FALSE)
if (have_old_boot) {
  old_dualscore$ibs_old_boot_same_method <- mapply(function(model, risk, horizon) {
    src <- if (model == "death_shap") saved$ibs_bp2 else saved$ibs_bp1
    old_boot_point(src, tolower(risk), horizon)
  }, old_dualscore$model, old_dualscore$risk, old_dualscore$horizon)
} else {
  old_dualscore$ibs_old_boot_same_method <- NA_real_
}

old_cif <- data.frame(model = character(0), horizon = numeric(0), ibs_old = numeric(0))
if (exists("ibs_cif_readmit_boot", envir = saved, inherits = FALSE)) {
  x <- saved$ibs_cif_readmit_boot
  cat("Found prior ibs_cif_readmit_boot in bundle; columns:", paste(names(x), collapse = ", "), "\n")
}

cat("\n=== OLD (pre-fix) dual-score IBS, from the bundle's own $summary ===\n")
print(old_dualscore, row.names = FALSE)

# ---- Load CORRECTED engine AFTER reading the old values ----
# evaluate_dual_cox_holdout_dualscore.R depends on ibs_ipcw_train()/to01()/etc. from
# evaluate_dual_cox_python_style_boot.R and guards on a bare exists(mode="function")
# that only looks in the caller's search path -- it does NOT see an isolated new.env(),
# so these are sourced directly here (into globalenv()), same as the project's own
# validate_holdout_metrics.R does. This is still safe against reusing the bundle's own
# stored copies: `saved` (loaded via load(bundle_file, envir=saved)) was never attached
# to the search path, so its old function copies stay completely inert and unreachable
# except via explicit saved$name access.
source(file.path(project_root, "cons/_alt_scripts/evaluate_dual_cox_python_style_boot.R"))
source(file.path(project_root, "cons/_alt_scripts/evaluate_dual_cox_holdout_dualscore.R"))
source(file.path(project_root, "cons/_alt_scripts/ibs_window_bootstrap_holdout.R"))
source(file.path(project_root, "cons/_alt_scripts/holdout_arm_comparison.R"))
stopifnot(exists("ibs_window_bootstrap_core", mode = "function"))
.body_text <- paste(deparse(body(ibs_ipcw_train)), collapse = " ")
stopifnot(grepl("match\\(requested_times, unique_times\\)", .body_text))
cat("\nCorrected engine sourced (globalenv(); the bundle's own old copies in `saved` stay unreachable).\n")

B_IBS <- 1000L
SEED_IBS <- 2125L
GRID_IBS <- c(3, 6, 12, 36, 60)

# ---- 1. Dual-score holdout IBS (readmission net risk + death, both models) ----
cat("\n=== Rescoring section 1: dual-score holdout IBS (B=1000) ===\n")
ibs_bp1 <- ibs_window_bootstrap_core(saved$results_boot_val_bp1, B = B_IBS, seed = SEED_IBS,
                                     eval_times = GRID_IBS, readmit_method = "ipcw",
                                     compute_null = TRUE, verbose = TRUE)
ibs_bp2 <- ibs_window_bootstrap_core(saved$results_boot_val_bp2, B = B_IBS, seed = SEED_IBS,
                                     eval_times = GRID_IBS, readmit_method = "ipcw",
                                     compute_null = TRUE, verbose = TRUE)

format_ibs <- function(x, model, risk) {
  observed <- x[x$risk == risk & x$horizon %in% HORIZONS, c("horizon", "point", "mean", "q025", "q975")]
  null <- attr(x, "null")
  null <- null[null$risk == risk & null$horizon %in% HORIZONS, c("horizon", "point", "q025", "q975")]
  stopifnot(identical(observed$horizon, null$horizon))
  data.frame(model = model, risk = risk, horizon = observed$horizon, ibs = observed$point,
             bootstrap_mean = observed$mean, lower = observed$q025, upper = observed$q975,
             ibs_null = null$point, null_lower = null$q025, null_upper = null$q975,
             ibs_skill = 1 - observed$point / null$point)
}
ibs_holdout_corrected <- rbind(
  format_ibs(ibs_bp1, "readmission_shared", "readmission"),
  format_ibs(ibs_bp1, "death_full_ph", "death"),
  format_ibs(ibs_bp2, "death_shap", "death")
)
write.csv(ibs_holdout_corrected, file.path(out_fix, "pred23_holdout_ibs_corrected_B1000.csv"), row.names = FALSE)
saveRDS(list(bp1 = ibs_bp1, bp2 = ibs_bp2), file.path(out_fix, "pred23_holdout_ibs_corrected_B1000.rds"))

# ---- 2. CIF-based readmission IBS (competing-risk aware), with skill vs the ----
#         no-predictor (marginal Aalen-Johansen) null. compute_null = TRUE is
#         required here: for readmit_method = "aalen-johansen",
#         ibs_window_bootstrap_core() computes this null internally via
#         .ibsb_marg_aj_surv() (ibs_window_bootstrap_holdout.R), the marginal
#         CIF fit with no predictors -- the competing-risks-compatible null
#         model the skill score needs. compute_null = FALSE (as in the very
#         first draft of this cell) silently skips it.
cat("\n=== Rescoring section 2: CIF-based readmission IBS (B=1000) ===\n")
run_cif_ibs <- function(x, model) {
  ans <- ibs_window_bootstrap_core(x, B = B_IBS, seed = SEED_IBS, eval_times = GRID_IBS,
                                   readmit_method = "aalen-johansen", compute_null = TRUE, verbose = TRUE)
  observed <- ans[ans$risk == "readmission" & ans$horizon %in% HORIZONS, c("horizon", "point", "mean", "q025", "q975")]
  null <- attr(ans, "null")
  null <- null[null$risk == "readmission" & null$horizon %in% HORIZONS, c("horizon", "point", "q025", "q975")]
  stopifnot(identical(observed$horizon, null$horizon))
  data.frame(model = model, risk = "readmission_CIF", horizon = observed$horizon, ibs = observed$point,
             bootstrap_mean = observed$mean, lower = observed$q025, upper = observed$q975,
             ibs_null = null$point, null_lower = null$q025, null_upper = null$q975,
             ibs_skill = 1 - observed$point / null$point,
             B = B_IBS, seed = SEED_IBS)
}
ibs_cif_corrected <- rbind(
  run_cif_ibs(saved$results_boot_val_bp1_cif, "cif_full_ph_mortality"),
  run_cif_ibs(saved$results_boot_val_bp2_cif, "cif_shap_mortality")
)
write.csv(ibs_cif_corrected, file.path(out_fix, "pred23_readmission_cif_ibs_corrected_B1000.csv"), row.names = FALSE)
saveRDS(ibs_cif_corrected, file.path(out_fix, "pred23_readmission_cif_ibs_corrected_B1000.rds"))

# ---- 3. Mandatory checks against Codex's documented oracle values ----
get_ibs <- function(tab, model, horizon) tab$ibs[tab$model == model & tab$horizon == horizon]
checks <- data.frame(
  label = c("readmission_shared@60", "death_full_ph@60", "death_shap@60", "cif_full_ph_mortality@60", "cif_shap_mortality@60"),
  computed = c(
    get_ibs(ibs_holdout_corrected, "readmission_shared", 60),
    get_ibs(ibs_holdout_corrected, "death_full_ph", 60),
    get_ibs(ibs_holdout_corrected, "death_shap", 60),
    get_ibs(ibs_cif_corrected, "cif_full_ph_mortality", 60),
    get_ibs(ibs_cif_corrected, "cif_shap_mortality", 60)
  ),
  oracle = c(0.13635936633871, 0.02202086153821, 0.02219337215150, 0.13499635714777, 0.13504008339621)
)
checks$abs_diff <- abs(checks$computed - checks$oracle)
checks$pass <- checks$abs_diff < 1e-10
cat("\n=== Oracle cross-check (Codex's documented reference values) ===\n")
print(checks, row.names = FALSE)
stopifnot(all(checks$pass))
message("PASS: point IBS values reproduce Codex's documented corrected engine output.")

# ---- Build the old-vs-new comparison table the user asked for ----
new_dualscore <- ibs_holdout_corrected[, c("model", "risk", "horizon", "ibs")]
names(new_dualscore)[4] <- "ibs_new"
cmp_dualscore <- merge(old_dualscore, new_dualscore, by = c("model", "risk", "horizon"))
cmp_dualscore$abs_change <- cmp_dualscore$ibs_new - cmp_dualscore$ibs_old
cmp_dualscore$rel_change_pct <- 100 * cmp_dualscore$abs_change / cmp_dualscore$ibs_old
cmp_dualscore <- cmp_dualscore[order(cmp_dualscore$model, cmp_dualscore$horizon), ]

write.csv(cmp_dualscore, file.path(out_fix, "pred23_holdout_ibs_before_after.csv"), row.names = FALSE)

cat("\n\n=== FINAL: dual-score IBS, before (bundle $summary) vs after (corrected engine) ===\n")
print(cmp_dualscore, row.names = FALSE)

cat("\n=== FINAL: CIF-based readmission IBS, corrected, with skill vs the no-predictor AJ null (no pre-fix same-estimand baseline stored) ===\n")
print(ibs_cif_corrected[, c("model", "risk", "horizon", "ibs", "lower", "upper", "ibs_null", "ibs_skill")], row.names = FALSE)

cat("\nOutputs written to:", out_fix, "\n")
cat(sprintf("\n[pred23_ibs_rescore] TOTAL elapsed = %.2f min\n", as.numeric(difftime(Sys.time(), t0, units = "mins"))))

In [ ]:
# Section 3 (added 2026-07-22): corrected three-arm readmission comparison
# (net_risk / cif_bp1 / cif_bp2) and the competing-risk null + Brier skill for
# readmission under the SAME CIF (Aalen-Johansen) estimand for all 3 arms.
# Reuses `saved`, the sourced engine, GRID_IBS/HORIZONS/SEED_IBS/B_IBS, and
# `ibs_cif_corrected` from the cell above -- run that cell first.

B_ARM <- 500L

# ---- net_risk's readmission predictions, re-scored under the CIF (AJ) estimand ----
# Section 1's "readmission_shared" IBS used readmit_method = "ipcw" (death as
# censoring). This keeps the SAME predicted 1-S(t) curves from
# results_boot_val_bp1 but scores them against the competing-risk-aware
# observed outcome, so it is comparable to cif_full_ph_mortality/cif_shap_mortality.
cat("\n=== net_risk readmission IBS under the CIF estimand (B=1000) ===\n")
net_risk_cif_ans <- ibs_window_bootstrap_core(saved$results_boot_val_bp1, B = B_IBS, seed = SEED_IBS,
                                              eval_times = GRID_IBS, readmit_method = "aalen-johansen",
                                              compute_null = TRUE, verbose = TRUE)
observed <- net_risk_cif_ans[net_risk_cif_ans$risk == "readmission" & net_risk_cif_ans$horizon %in% HORIZONS,
                             c("horizon", "point", "mean", "q025", "q975")]
null <- attr(net_risk_cif_ans, "null")
null <- null[null$risk == "readmission" & null$horizon %in% HORIZONS, c("horizon", "point", "q025", "q975")]
stopifnot(identical(observed$horizon, null$horizon))
net_risk_under_cif <- data.frame(
  model = "net_risk", risk = "readmission_CIF", horizon = observed$horizon, ibs = observed$point,
  bootstrap_mean = observed$mean, lower = observed$q025, upper = observed$q975,
  ibs_null = null$point, null_lower = null$q025, null_upper = null$q975,
  ibs_skill = 1 - observed$point / null$point, B = B_IBS, seed = SEED_IBS
)

readmission_cif_skill_3arm <- rbind(net_risk_under_cif[, names(ibs_cif_corrected)], ibs_cif_corrected)
readmission_cif_skill_3arm <- readmission_cif_skill_3arm[order(readmission_cif_skill_3arm$model, readmission_cif_skill_3arm$horizon), ]
write.csv(readmission_cif_skill_3arm, file.path(out_fix, "pred23_readmission_cif_ibs_3arm_null_skill_B1000.csv"), row.names = FALSE)

cat("\n=== Readmission IBS/null/skill, all 3 arms, SAME CIF (Aalen-Johansen) estimand ===\n")
print(readmission_cif_skill_3arm[, c("model", "horizon", "ibs", "ibs_null", "ibs_skill")], row.names = FALSE)

# ---- Corrected pairwise three-arm comparison (net_risk vs cif_bp1 vs cif_bp2) ----
cat("\n=== Corrected 3-arm pairwise IBS comparison (B=500) ===\n")
arms_readmission <- list(
  net_risk = saved$results_boot_val_bp1,
  cif_bp1 = saved$results_boot_val_bp1_cif,
  cif_bp2 = saved$results_boot_val_bp2_cif
)
ibs_arm_comparison <- .holdout_compare_ibs(
  arms_readmission, eval_times = GRID_IBS, times = HORIZONS,
  readmit_method = "aalen-johansen", B = B_ARM, seed = SEED_IBS
)
write.csv(ibs_arm_comparison, file.path(out_fix, "pred23_holdout_arm_comparison_ibs_corrected_B500.csv"), row.names = FALSE)
saveRDS(ibs_arm_comparison, file.path(out_fix, "pred23_holdout_arm_comparison_ibs_corrected_B500.rds"))

cat("\n=== 3-arm pairwise IBS comparison (net_risk / cif_bp1 / cif_bp2), same CIF estimand ===\n")
print(ibs_arm_comparison[, c("arm_A", "arm_B", "horizon", "mean_A", "mean_B", "diff", "ij_lower", "ij_upper", "excludes_zero_ij")], row.names = FALSE)

# Cross-check: .holdout_compare_ibs()'s mean_A for net_risk must equal
# ibs_window_bootstrap_core()'s point estimate for net_risk at the same
# horizon/estimand -- the documented invariant from holdout_arm_comparison.R's
# own header comment (both are the same per-patient contribution mean,
# computed by two different code paths).
check_rows <- ibs_arm_comparison[ibs_arm_comparison$arm_A == "net_risk", ]
for (h in HORIZONS) {
  row1 <- check_rows[check_rows$horizon == h, ]
  ref <- net_risk_under_cif$ibs[net_risk_under_cif$horizon == h]
  if (nrow(row1)) stopifnot(abs(row1$mean_A[1] - ref) < 1e-8)
}
cat("\nCheck PASS: arm-comparison's net_risk point IBS matches ibs_window_bootstrap_core()'s net_risk-under-CIF point IBS.\n")

In [ ]:
# Section 4 (added 2026-07-22): rescore Sens/Spec/PPV/NPV at fixed horizons.
# The stored thr_boot/thr_boot_death/thr_boot_readmit in this bundle used
# bootstrap_threshold_metrics_holdout() (cons/_alt_scripts/validate_holdout_metrics.R),
# which reuses .tm_admin_ipcw_weights()/.tm_cr_ipcw_weights() from
# threshold_metrics_from_results_boot.R. That file was patched for the same
# order-preserving IPCW lookup fix on 2026-07-21 (confirmed by mtime and by
# .tm_survfit_lookup_ordered()'s implementation), AFTER this bundle was saved
# on 2026-07-19, so the stored values are stale. Reuses `saved` and `project_root`
# from the cells above; run those first.

stopifnot(exists("thr_boot", envir = saved, inherits = FALSE))
old_thr_boot <- saved$thr_boot

source(file.path(project_root, "cons/_alt_scripts/threshold_metrics_from_results_boot.R"))
stopifnot(exists(".tm_confusion", mode = "function"), exists(".tm_cr_ipcw_weights", mode = "function"))
.body_text <- paste(deparse(body(.tm_survfit_lookup_ordered)), collapse = " ")
stopifnot(grepl("match\\(requested_times\\[finite\\], unique_times\\)", .body_text))

# bootstrap_threshold_metrics_holdout() / threshold_bootstrap_holdout(), copied
# verbatim from cons/_alt_scripts/validate_holdout_metrics.R (lines 531-740).
# Only these two functions are copied (not the whole 740-line file, which also
# declares riskRegression/prodlim dependencies unrelated to threshold metrics);
# both call ONLY .tm_confusion/.tm_cr_ipcw_weights/.tm_admin_ipcw_weights/
# .tm_cr_reconstruct, all from the file sourced above.
bootstrap_threshold_metrics_holdout <- function(
    results_boot_val, model_label = "model",
    risks = c("readmission", "death"),
    horizons = c(6, 12, 36, 60),
    thresholds = list(readmission = c(0.10, 0.15, 0.20, 0.25, 0.30, 0.40),
                      death       = c(0.01, 0.02, 0.03, 0.05, 0.075, 0.10)),
    metrics = c("Sens", "Spec", "PPV", "NPV"),
    central = c("plugin", "median", "mean"),
    ci_method = c("percentile", "bca"),
    freeze_g = TRUE,
    B = 500L, seed = 2125L, g_min = 0.05, tie_action = "death_first",
    parallel = FALSE, verbose = TRUE) {

  central <- match.arg(central)
  ci_method <- match.arg(ci_method)
  if (!exists(".tm_confusion", mode = "function") ||
      !exists(".tm_cr_ipcw_weights", mode = "function"))
    source(file.path(project_root, "cons/_alt_scripts/threshold_metrics_from_results_boot.R"))

  rp <- results_boot_val$raw_predictions
  eval_times <- as.numeric(results_boot_val$config$eval_times %||% rp[[1]]$eval_times)

  pool <- list()
  for (risk in risks) {
    blk <- Filter(function(b) is.list(b) && is.null(b$error) && !is.null(b$surv_val_matrix),
                  lapply(rp, function(it) it[[risk]]))
    nrs <- vapply(blk, function(b) nrow(as.matrix(b$surv_val_matrix)), integer(1))
    if (length(unique(nrs)) != 1L)
      stop("Imputation blocks for '", risk, "' have different row counts; cannot MI-pool.", call. = FALSE)
    surv_sum <- Reduce(`+`, lapply(blk, function(b) as.matrix(b$surv_val_matrix)))
    pred_mat <- 1 - surv_sum / length(blk)
    yv <- rp[[1]][[risk]]$y_val
    if (risk == "readmission") {
      dv <- rp[[1]][["death"]]$y_val
      cr <- .tm_cr_reconstruct(yv$time, yv$event, dv$time, dv$event, tie_action = tie_action)
      pool[[risk]] <- list(pred = pred_mat, ftime = cr$ftime, fstatus = cr$fstatus, type = "cr")
    } else {
      pool[[risk]] <- list(pred = pred_mat, time = as.numeric(yv$time),
                           event = as.integer(yv$event), type = "km")
    }
  }
  n <- nrow(pool[[risks[1]]]$pred)

  Wfull <- list()
  if (isTRUE(freeze_g)) {
    for (risk in risks) {
      P <- pool[[risk]]
      for (h in horizons) {
        hj <- which(eval_times == h); if (!length(hj)) next
        Wfull[[paste(risk, h, sep = "|")]] <-
          if (P$type == "cr") .tm_cr_ipcw_weights(P$ftime, P$fstatus, h, g_min = g_min)
          else                .tm_admin_ipcw_weights(P$time, P$event, h, g_min = g_min)
      }
    }
  }

  one_rep <- function(idx) {
    out <- list()
    for (risk in risks) {
      P <- pool[[risk]]; thr <- thresholds[[risk]]
      for (h in horizons) {
        hj <- which(eval_times == h); if (!length(hj)) next
        pr <- P$pred[idx, hj[1]]
        if (isTRUE(freeze_g)) {
          wf <- Wfull[[paste(risk, h, sep = "|")]]
          we <- wf$w_event[idx]; wn <- wf$w_nonevent[idx]
        } else {
          w <- if (P$type == "cr") .tm_cr_ipcw_weights(P$ftime[idx], P$fstatus[idx], h, g_min = g_min)
               else                .tm_admin_ipcw_weights(P$time[idx], P$event[idx], h, g_min = g_min)
          we <- w$w_event; wn <- w$w_nonevent
        }
        for (t in thr) {
          cm <- .tm_confusion(pr, t, we, wn)
          out[[length(out) + 1L]] <- data.frame(risk = risk, horizon = h, threshold = t,
            as.list(cm[metrics]), check.names = FALSE, stringsAsFactors = FALSE)
        }
      }
    }
    do.call(rbind, out)
  }

  point <- one_rep(seq_len(n))
  rep_fun <- function(b) {
    set.seed(seed + b, kind = "Mersenne-Twister")
    one_rep(sample.int(n, n, replace = TRUE))
  }
  use_par <- isTRUE(parallel) && requireNamespace("future.apply", quietly = TRUE)
  if (verbose) cat(sprintf("threshold bootstrap [%s]: B=%d, parallel=%s\n", model_label, B, use_par))
  boots <- if (use_par)
    future.apply::future_lapply(seq_len(B), rep_fun, future.seed = TRUE,
      future.packages = c("survival"))
  else lapply(seq_len(B), rep_fun)

  allb <- do.call(rbind, boots)
  k_all <- paste(allb$risk, allb$horizon, allb$threshold, sep = "|")
  k_pt  <- paste(point$risk, point$horizon, point$threshold, sep = "|")
  res <- point
  bq    <- function(mt, p) as.numeric(tapply(allb[[mt]], k_all,
             function(x) stats::quantile(x, p, na.rm = TRUE, names = FALSE))[k_pt])
  bmean <- function(mt)    as.numeric(tapply(allb[[mt]], k_all,
             function(x) mean(x, na.rm = TRUE))[k_pt])
  for (mt in metrics) {
    res[[paste0(mt, "_plugin")]] <- res[[mt]]
    res[[paste0(mt, "_med")]]    <- bq(mt, 0.5)
    res[[paste0(mt, "_mean")]]   <- bmean(mt)
    res[[paste0(mt, "_lo")]]     <- bq(mt, 0.025)
    res[[paste0(mt, "_hi")]]     <- bq(mt, 0.975)
    res[[mt]] <- switch(central,
                        plugin = res[[paste0(mt, "_plugin")]],
                        median = res[[paste0(mt, "_med")]],
                        mean   = res[[paste0(mt, "_mean")]])
  }
  attr(res, "ci_method") <- ci_method
  res$model <- model_label
  res <- res[order(res$risk, res$horizon, res$threshold), ]
  rownames(res) <- NULL
  res
}

threshold_bootstrap_holdout <- function(results_boot_val, horizons = c(6, 12, 36, 60),
                                        thresholds = NULL, metrics = c("Sens", "Spec", "PPV", "NPV"),
                                        central = c("plugin", "median", "mean"),
                                        ci_method = c("percentile", "bca"),
                                        freeze_g = TRUE,
                                        B = 500L, seed = 2125L, parallel = FALSE, verbose = TRUE) {
  central <- match.arg(central); ci_method <- match.arg(ci_method)
  args <- list(horizons = horizons, metrics = metrics, central = central, ci_method = ci_method,
               freeze_g = freeze_g, B = B, seed = seed, parallel = parallel, verbose = verbose)
  if (!is.null(thresholds)) args$thresholds <- thresholds
  out <- list()
  out[[1]] <- do.call(bootstrap_threshold_metrics_holdout,
    c(list(results_boot_val[[1]], model_label = "readmit::shared", risks = "readmission"), args))
  for (mn in names(results_boot_val))
    out[[length(out) + 1L]] <- do.call(bootstrap_threshold_metrics_holdout,
      c(list(results_boot_val[[mn]], model_label = paste0("death::", mn), risks = "death"), args))
  do.call(rbind, out)
}

# ---- Re-run EXACTLY the original notebook cell 'holdout-threshold-bootstrap-run' ----
cat("\n=== Rescoring: thr_boot_death (readmit::shared discarded, death::best_perf1/2 kept) ===\n")
thr_boot_death <- threshold_bootstrap_holdout(
  list(best_perf1 = saved$results_boot_val_bp1, best_perf2 = saved$results_boot_val_bp2),
  horizons = c(6, 12, 36, 60), central = "plugin", ci_method = "percentile",
  freeze_g = TRUE, B = 1000L, seed = 2125L, parallel = TRUE, verbose = TRUE
)

cat("\n=== Rescoring: thr_boot_readmit (netrisk / bp1_cif / bp2_cif) ===\n")
thr_boot_readmit <- rbind(
  bootstrap_threshold_metrics_holdout(saved$results_boot_val_bp1,     model_label = "readmit::netrisk", risks = "readmission", horizons = c(6,12,36,60), central = "plugin", ci_method = "percentile", freeze_g = TRUE, B = 1000L, seed = 2125L, parallel = TRUE, verbose = TRUE),
  bootstrap_threshold_metrics_holdout(saved$results_boot_val_bp1_cif, model_label = "readmit::bp1_cif",  risks = "readmission", horizons = c(6,12,36,60), central = "plugin", ci_method = "percentile", freeze_g = TRUE, B = 1000L, seed = 2125L, parallel = TRUE, verbose = TRUE),
  bootstrap_threshold_metrics_holdout(saved$results_boot_val_bp2_cif, model_label = "readmit::bp2_cif",  risks = "readmission", horizons = c(6,12,36,60), central = "plugin", ci_method = "percentile", freeze_g = TRUE, B = 1000L, seed = 2125L, parallel = TRUE, verbose = TRUE)
)

thr_boot <- rbind(thr_boot_readmit[thr_boot_readmit$risk == "readmission", ], thr_boot_death[thr_boot_death$risk == "death", ])

write.csv(thr_boot, file.path(out_fix, "pred23_holdout_threshold_bootstrap_corrected.csv"), row.names = FALSE)
saveRDS(list(thr_boot = thr_boot, thr_boot_death = thr_boot_death, thr_boot_readmit = thr_boot_readmit),
        file.path(out_fix, "pred23_holdout_threshold_bootstrap_corrected.rds"))

# ---- Before vs after comparison ----
key_cols <- c("model", "risk", "horizon", "threshold")
metric_cols <- c("Sens", "Spec", "PPV", "NPV")
old_sel <- old_thr_boot[, c(key_cols, metric_cols, paste0(metric_cols, "_lo"), paste0(metric_cols, "_hi"))]
names(old_sel)[-(1:4)] <- paste0(names(old_sel)[-(1:4)], "_old")
new_sel <- thr_boot[, c(key_cols, metric_cols, paste0(metric_cols, "_lo"), paste0(metric_cols, "_hi"))]
names(new_sel)[-(1:4)] <- paste0(names(new_sel)[-(1:4)], "_new")

cmp_thr <- merge(old_sel, new_sel, by = key_cols)
stopifnot(nrow(cmp_thr) == nrow(old_thr_boot), nrow(cmp_thr) == nrow(thr_boot))
cmp_thr <- cmp_thr[order(cmp_thr$risk, cmp_thr$model, cmp_thr$horizon, cmp_thr$threshold), ]
write.csv(cmp_thr, file.path(out_fix, "pred23_holdout_threshold_before_after.csv"), row.names = FALSE)

cat("\n=== Threshold metrics before vs after: max |Spec change| by risk (mechanistic check; ----\n")
cat("     death uses a purely scalar admin-censoring lookup so Spec should be ~0 change;\n")
cat("     readmission's competing-risk weights have a vector-dependent branch, so a small\n")
cat("     change there is expected, not a bug) ===\n")
cmp_thr$Spec_diff <- cmp_thr$Spec_new - cmp_thr$Spec_old
print(aggregate(Spec_diff ~ risk, cmp_thr, function(x) max(abs(x))))

cat("\nFull before/after table (120 rows): pred23_holdout_threshold_before_after.csv\n")
cat(sprintf("\n[section 4: threshold rescoring] done\n"))

## Verification performed before delivery

- The 3 engine scripts were checked to confirm the corrected, order-preserving
  IPCW lookup is used (`match(requested_times, unique_times)` present in
  `ibs_ipcw_train()` and `.ibsb_G_fun()`), not a legacy version.
- `evaluate_dual_cox_holdout_dualscore.R` has an internal dependency guard
  that only sees globalenv(), not an isolated `new.env()`; the original
  suggested code isolated the engine in `new.env()`, which fails that guard.
  Fixed here by sourcing directly (the loaded bundle `saved` never touches
  globalenv(), so the bundle's own old function copies stay unreachable
  regardless).
- All 5 point-IBS values at horizon 60 were checked against documented
  reference values, abs diff < 1e-10 for all (actual: ~1e-15, floating-point
  noise).
- The "old" values used for the before/after comparison come from the same
  loaded bundle's own `$summary` (Metric == "IBS"), so the comparison is
  apples-to-apples: same models, same horizons, same held-out patients.

- Section 4 rescores the fixed-horizon threshold metrics
  (`bootstrap_threshold_metrics_holdout()` / `threshold_bootstrap_holdout()`,
  `cons/_alt_scripts/validate_holdout_metrics.R`). That file's own mtime
  predates the July 21 fix, but it reuses (does not reimplement)
  `.tm_admin_ipcw_weights()`/`.tm_cr_ipcw_weights()` from
  `threshold_metrics_from_results_boot.R`, which WAS patched (confirmed by
  mtime and by `.tm_survfit_lookup_ordered()`), so it inherits the fix once
  that file is sourced first. As a mechanistic cross-check: death's
  Specificity comes out bit-identical before/after (its weights use only a
  scalar horizon lookup, which the bug could not corrupt), while
  readmission's Specificity shifts slightly (its competing-risk weights have
  a vector-dependent branch for competing deaths before the horizon) --
  exactly the pattern the bug's mechanism predicts, not noise.

**Scope note.** Per the request, this notebook does not touch imputation,
data splitting, Cox refitting, SHAP, AICc, calibration, DCA, NRI/IDI, or
`prediction24`. Section 3 adds the corrected three-arm readmission comparison
(net_risk / cif_bp1 / cif_bp2) and the competing-risk (Aalen-Johansen,
no-predictor) null and Brier skill for all three arms under the same CIF
estimand, cross-checked against the independent point-estimate computation
in section 2's engine. Uno's C (`.holdout_compare_predicted_prob`,
`.holdout_compare_wolbers`) was left out; it is a distinct metric family, not
requested.